# CSE498R Final Thesis Notebook — Stage 2 Improved V2  
**Topic:** Improved Advanced Inference Pipeline  
**Pipeline:** YOLO Segmentation → Refined Removal Mask → LaMa Global Inpainting → Stable Diffusion Local Refinement → Improved Feather Blending → Residual Detection Check

This notebook is the **improved V2 notebook** built from your working Stage 2 advanced pipeline.

## What is improved in V2
1. refined removal mask with morphological cleanup  
2. smoother feather blending for local patch fusion  
3. separate experiment folder for clean comparison  
4. safer inference loop and logging  

## Goal
Use this notebook to compare against the earlier Stage 2 baseline and check whether the improved mask + blending reduces artifacts and improves realism.


In [ ]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
# Run this cell once, then manually restart the runtime.
# After restart, do NOT run this cell again in the same session.

!pip uninstall -y numpy lpips basemap opencv-python opencv-python-headless opencv-contrib-python
!pip cache purge

!pip install -q numpy==1.26.4
!pip install -q ultralytics
!pip install -q simple-lama-inpainting
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python-headless==4.10.0.84
!pip install -q pillow tqdm pandas matplotlib scikit-image
!pip install -q lpips==0.1.4

print("INSTALL CELL FINISHED — NOW RESTART RUNTIME MANUALLY")


In [ ]:
# ============================================================
# CELL 2: Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CSE498R_Final_Thesis")
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"
SPLIT_ROOT = DATA_ROOT / "splits"
RESULT_ROOT = PROJECT_ROOT / "results"
MODEL_ROOT = PROJECT_ROOT / "models"
VIS_ROOT = PROJECT_ROOT / "visualizations"

EXPERIMENT_NAME = "stage2_improved_v2"
STAGE2_ROOT = RESULT_ROOT / EXPERIMENT_NAME
STAGE2_IMG_ROOT = STAGE2_ROOT / "images"
STAGE2_META_ROOT = STAGE2_ROOT / "metadata"

for p in [RESULT_ROOT, MODEL_ROOT, VIS_ROOT, STAGE2_ROOT, STAGE2_IMG_ROOT, STAGE2_META_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Stage 2 result paths ready.")
print("EXPERIMENT_NAME =", EXPERIMENT_NAME)
print("STAGE2_ROOT =", STAGE2_ROOT)


In [ ]:
# ============================================================
# CELL 3: Imports
# ============================================================

import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

import cv2
import lpips
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from PIL import Image
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from simple_lama_inpainting import SimpleLama
from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)


In [ ]:
# ============================================================
# CELL 4: Configuration
# ============================================================

CFG = {
    "target_split": "test",
    "max_samples": None,

    # Final mask settings
    "use_context_ring": True,
    "mask_extra_dilate_px": 6,
    "mask_smooth_kernel": 11,
    "mask_close_kernel": 9,
    "mask_open_kernel": 5,

    # Local crop and blend settings
    "local_crop_expand_ratio": 1.85,
    "local_target_size": 512,
    "feather_blur_ksize": 31,
    "local_blend_strength": 1.0,

    # Residual detector
    "detector_model": "yolov8x-seg.pt",
    "residual_conf_threshold": 0.20,
    "residual_class_name": "person",

    # Local diffusion prompt
    "positive_prompt": "natural clean background, high detail, realistic texture, seamless scene, photorealistic, no people, no humans",
    "negative_prompt": "person, human, face, body, blur, artifacts, distortion, ghost, shadow, duplicate, watermark, logo",

    # Local diffusion inpaint model
    "local_inpaint_model_id": "sd2-community/stable-diffusion-2-inpainting",
    "local_num_inference_steps": 30,
    "local_guidance_scale": 7.5,
    "local_strength_fallback_to_lama": True,

    # Visualization
    "num_preview_samples": 5
}

print(json.dumps(CFG, indent=2))



## Data files expected from previous stages

This notebook expects these files to already exist:

- `data/processed/images/*.png`
- `data/processed/masks/*.png`
- `data/processed/context_rings/*.png`
- `data/splits/split_metadata.csv`


In [ ]:
# ============================================================
# CELL 5: Load split metadata
# ============================================================

split_csv = SPLIT_ROOT / "split_metadata.csv"
df = pd.read_csv(split_csv)

df_target = df[df["split"] == CFG["target_split"]].copy().reset_index(drop=True)

if CFG["max_samples"] is not None:
    df_target = df_target.head(CFG["max_samples"]).copy()

print("Loaded split file:", split_csv)
print("Target split:", CFG["target_split"])
print("Samples to process:", len(df_target))
display(df_target.head())


In [ ]:
# ============================================================
# CELL 6: Load main models
# ============================================================

# 6A. Global inpainting model: LaMa
simple_lama = SimpleLama()
print("Global inpainting model (LaMa) ready.")

# 6B. Residual detector
detector = YOLO(CFG["detector_model"])
print("Residual detector ready:", CFG["detector_model"])

# 6C. Local diffusion inpainting pipeline
# Corrected loading code for diffusers inpainting pipeline.
# If loading fails, the notebook will still continue with LaMa fallback.

local_pipe = None
local_pipe_name = None

try:
    from diffusers import AutoPipelineForInpainting

    if DEVICE == "cuda":
        local_pipe = AutoPipelineForInpainting.from_pretrained(
            CFG["local_inpaint_model_id"],
            torch_dtype=torch.float16,
            variant="fp16"
        )
        local_pipe = local_pipe.to("cuda")
    else:
        local_pipe = AutoPipelineForInpainting.from_pretrained(
            CFG["local_inpaint_model_id"],
            torch_dtype=torch.float32
        )

    # Optional memory optimization
    try:
        local_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    # Optional attention slicing
    try:
        local_pipe.enable_attention_slicing()
    except Exception:
        pass

    local_pipe_name = CFG["local_inpaint_model_id"]
    print("Local diffusion pipeline loaded:", local_pipe_name)

except Exception as e:
    print("Could not load local diffusion inpainting pipeline.")
    print("Reason:", e)
    print("Fallback mode will use local LaMa refinement only.")


In [ ]:
# ============================================================
# CELL 7: Metric model
# ============================================================

lpips_model = lpips.LPIPS(net='alex').to(DEVICE)
print("LPIPS model ready.")


In [ ]:
# ============================================================
# CELL 8: Helper functions
# ============================================================

def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def read_gray(path):
    return np.array(Image.open(path).convert("L"))

def to_pil_rgb(arr):
    return Image.fromarray(arr.astype(np.uint8)).convert("RGB")

def to_pil_gray(arr):
    return Image.fromarray(arr.astype(np.uint8)).convert("L")

def binary_mask(arr):
    return ((arr > 0).astype(np.uint8) * 255)

def dilate_mask(mask, pixels):
    if pixels <= 0:
        return binary_mask(mask)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (pixels * 2 + 1, pixels * 2 + 1))
    out = cv2.dilate(binary_mask(mask), kernel, iterations=1)
    return binary_mask(out)

def merge_mask_and_ring(mask, ring, use_context_ring=True, extra_dilate_px=0):
    base = binary_mask(mask)
    if use_context_ring:
        base = np.maximum(base, binary_mask(ring))
    if extra_dilate_px > 0:
        base = dilate_mask(base, extra_dilate_px)
    return binary_mask(base)

def refine_removal_mask(mask, ring=None, extra_dilate_px=0,
                        smooth_kernel=11, close_kernel=9, open_kernel=5):
    """
    Refine the final removal mask to reduce rough edges and small artifacts.
    Steps:
    1. merge core mask and optional context ring
    2. morphological close to fill narrow gaps
    3. morphological open to remove tiny noise
    4. optional dilation
    5. soft smoothing + threshold back to binary
    """
    base = binary_mask(mask)

    if ring is not None:
        base = np.maximum(base, binary_mask(ring))

    if close_kernel > 1:
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        base = cv2.morphologyEx(base, cv2.MORPH_CLOSE, k_close)

    if open_kernel > 1:
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
        base = cv2.morphologyEx(base, cv2.MORPH_OPEN, k_open)

    if extra_dilate_px > 0:
        base = dilate_mask(base, extra_dilate_px)

    if smooth_kernel > 1:
        if smooth_kernel % 2 == 0:
            smooth_kernel += 1
        soft = cv2.GaussianBlur(base, (smooth_kernel, smooth_kernel), 0)
        base = ((soft > 127).astype(np.uint8) * 255)

    return base

def make_soft_alpha(mask, blur_ksize=31):
    """Create a soft alpha matte from a binary mask for smoother blending."""
    alpha = (binary_mask(mask) > 0).astype(np.float32)

    if blur_ksize > 1:
        if blur_ksize % 2 == 0:
            blur_ksize += 1
        alpha = cv2.GaussianBlur(alpha, (blur_ksize, blur_ksize), 0)

    alpha = np.clip(alpha, 0.0, 1.0)
    return alpha

def get_bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x1, y1 = int(xs.min()), int(ys.min())
    x2, y2 = int(xs.max()) + 1, int(ys.max()) + 1
    return [x1, y1, x2, y2]

def expand_bbox_to_square(bbox, image_shape, expand_ratio=1.85):
    h, w = image_shape[:2]
    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0

    side = max(bw, bh) * expand_ratio
    side = max(side, 64)

    nx1 = int(round(cx - side / 2))
    ny1 = int(round(cy - side / 2))
    nx2 = int(round(cx + side / 2))
    ny2 = int(round(cy + side / 2))

    nx1 = max(0, nx1)
    ny1 = max(0, ny1)
    nx2 = min(w, nx2)
    ny2 = min(h, ny2)

    return [nx1, ny1, nx2, ny2]

def crop_by_bbox(image, bbox):
    x1, y1, x2, y2 = bbox
    return image[y1:y2, x1:x2].copy()

def smooth_blend_patch(global_image, local_patch, local_mask, bbox,
                       blur_ksize=31, blend_strength=1.0):
    """
    Improved local blending:
    - resize patch and mask safely
    - create soft alpha from mask
    - blend more smoothly into the global image
    """
    x1, y1, x2, y2 = bbox
    out = global_image.copy()

    target_h = y2 - y1
    target_w = x2 - x1

    patch = cv2.resize(local_patch, (target_w, target_h), interpolation=cv2.INTER_CUBIC)
    alpha = make_soft_alpha(local_mask, blur_ksize=blur_ksize)
    alpha = cv2.resize(alpha, (target_w, target_h), interpolation=cv2.INTER_CUBIC)

    alpha = np.clip(alpha * blend_strength, 0.0, 1.0)[..., None]

    base_crop = out[y1:y2, x1:x2].astype(np.float32)
    patch = patch.astype(np.float32)

    blended = alpha * patch + (1.0 - alpha) * base_crop
    out[y1:y2, x1:x2] = np.clip(blended, 0, 255).astype(np.uint8)

    return out

def run_lama(image_rgb, mask_gray):
    out = simple_lama(to_pil_rgb(image_rgb), to_pil_gray(binary_mask(mask_gray)))
    return np.array(out.convert("RGB"))

def run_local_diffusion_or_fallback(local_image_rgb, local_mask_gray):
    if local_pipe is None:
        return run_lama(local_image_rgb, local_mask_gray), "lama_local_fallback"

    try:
        image_pil = to_pil_rgb(local_image_rgb).resize((CFG["local_target_size"], CFG["local_target_size"]))
        mask_pil = to_pil_gray(binary_mask(local_mask_gray)).resize((CFG["local_target_size"], CFG["local_target_size"]))

        with torch.no_grad():
            result = local_pipe(
                prompt=CFG["positive_prompt"],
                negative_prompt=CFG["negative_prompt"],
                image=image_pil,
                mask_image=mask_pil,
                num_inference_steps=CFG["local_num_inference_steps"],
                guidance_scale=CFG["local_guidance_scale"]
            ).images[0]

        return np.array(result.convert("RGB")), "diffusion_local"

    except Exception as e:
        print("Local diffusion failed, using local LaMa fallback. Reason:", e)
        if CFG["local_strength_fallback_to_lama"]:
            return run_lama(local_image_rgb, local_mask_gray), "lama_local_fallback"
        raise

def detect_residual_persons(image_rgb):
    result = detector.predict(
        source=image_rgb,
        conf=CFG["residual_conf_threshold"],
        verbose=False
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        return 0, 0.0

    residual_class_id = None
    for k, v in detector.names.items():
        if str(v).lower() == CFG["residual_class_name"]:
            residual_class_id = int(k)
            break

    if residual_class_id is None:
        return 0, 0.0

    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    confs = result.boxes.conf.detach().cpu().numpy().astype(float)

    selected_confs = [float(c) for c, cl in zip(confs, classes) if cl == residual_class_id]

    if len(selected_confs) == 0:
        return 0, 0.0

    return len(selected_confs), max(selected_confs)

def np_to_tensor_lpips(img_rgb):
    x = torch.from_numpy(img_rgb).float() / 127.5 - 1.0
    x = x.permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    return x

def compute_lpips(img1_rgb, img2_rgb):
    if img1_rgb.shape != img2_rgb.shape:
        img2_rgb = cv2.resize(img2_rgb, (img1_rgb.shape[1], img1_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    with torch.no_grad():
        t1 = np_to_tensor_lpips(img1_rgb)
        t2 = np_to_tensor_lpips(img2_rgb)
        score = lpips_model(t1, t2).item()
    return float(score)

def compute_region_metrics(reference_rgb, predicted_rgb, region_mask):
    """
    Safe metric computation.
    It automatically fixes shape mismatch before PSNR / SSIM / LPIPS.
    """

    region_mask = binary_mask(region_mask)

    # Make predicted image same size as reference if needed
    if predicted_rgb.shape[:2] != reference_rgb.shape[:2]:
        predicted_rgb = cv2.resize(
            predicted_rgb,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    # Make mask same size as reference if needed
    if region_mask.shape[:2] != reference_rgb.shape[:2]:
        region_mask = cv2.resize(
            region_mask,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    bbox = get_bbox_from_mask(region_mask)
    if bbox is None:
        return {
            "psnr_region": np.nan,
            "ssim_region": np.nan,
            "lpips_full": np.nan,
            "out_ssim": np.nan
        }

    x1, y1, x2, y2 = bbox

    ref_crop = reference_rgb[y1:y2, x1:x2]
    pred_crop = predicted_rgb[y1:y2, x1:x2]

    # Final safety check
    if ref_crop.shape[:2] != pred_crop.shape[:2]:
        pred_crop = cv2.resize(
            pred_crop,
            (ref_crop.shape[1], ref_crop.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    # If crop too small, SSIM may fail
    if ref_crop.shape[0] < 7 or ref_crop.shape[1] < 7:
        ssim_region = np.nan
    else:
        ssim_region = structural_similarity(
            ref_crop,
            pred_crop,
            channel_axis=2,
            data_range=255
        )

    psnr_region = peak_signal_noise_ratio(ref_crop, pred_crop, data_range=255)
    lpips_full = compute_lpips(reference_rgb, predicted_rgb)

    if reference_rgb.shape[:2] != predicted_rgb.shape[:2]:
        predicted_rgb = cv2.resize(
            predicted_rgb,
            (reference_rgb.shape[1], reference_rgb.shape[0]),
            interpolation=cv2.INTER_CUBIC
        )

    if reference_rgb.shape[0] < 7 or reference_rgb.shape[1] < 7:
        out_ssim = np.nan
    else:
        out_ssim = structural_similarity(
            reference_rgb,
            predicted_rgb,
            channel_axis=2,
            data_range=255
        )

    return {
        "psnr_region": float(psnr_region),
        "ssim_region": float(ssim_region) if not np.isnan(ssim_region) else np.nan,
        "lpips_full": float(lpips_full),
        "out_ssim": float(out_ssim) if not np.isnan(out_ssim) else np.nan
    }


In [ ]:
# ============================================================
# CELL 9: One-sample pipeline function
# ============================================================

def process_one_sample(row):
    image = read_rgb(row["image_path"])
    mask = read_gray(row["mask_path"])
    ring = read_gray(row["ring_path"])

    if CFG["use_context_ring"]:
        final_mask = refine_removal_mask(
            mask=mask,
            ring=ring,
            extra_dilate_px=CFG["mask_extra_dilate_px"],
            smooth_kernel=CFG["mask_smooth_kernel"],
            close_kernel=CFG["mask_close_kernel"],
            open_kernel=CFG["mask_open_kernel"]
        )
    else:
        final_mask = refine_removal_mask(
            mask=mask,
            ring=None,
            extra_dilate_px=CFG["mask_extra_dilate_px"],
            smooth_kernel=CFG["mask_smooth_kernel"],
            close_kernel=CFG["mask_close_kernel"],
            open_kernel=CFG["mask_open_kernel"]
        )

    t0 = time.time()
    global_result = run_lama(image, final_mask)
    t_global = time.time() - t0

    bbox = get_bbox_from_mask(final_mask)
    if bbox is None:
        return None

    local_bbox = expand_bbox_to_square(
        bbox=bbox,
        image_shape=image.shape,
        expand_ratio=CFG["local_crop_expand_ratio"]
    )

    local_image = crop_by_bbox(global_result, local_bbox)
    local_mask = crop_by_bbox(final_mask, local_bbox)

    t1 = time.time()
    local_result, local_method = run_local_diffusion_or_fallback(local_image, local_mask)
    t_local = time.time() - t1

    fused_result = smooth_blend_patch(
        global_image=global_result,
        local_patch=local_result,
        local_mask=local_mask,
        bbox=local_bbox,
        blur_ksize=CFG["feather_blur_ksize"],
        blend_strength=CFG["local_blend_strength"]
    )

    residual_count, residual_max_conf = detect_residual_persons(fused_result)

    second_pass_used = False
    if residual_count > 0:
        second_pass_used = True
        local_mask_2 = dilate_mask(local_mask, 8)
        local_result_2, _ = run_local_diffusion_or_fallback(local_image, local_mask_2)

        fused_result = smooth_blend_patch(
            global_image=fused_result,
            local_patch=local_result_2,
            local_mask=local_mask_2,
            bbox=local_bbox,
            blur_ksize=CFG["feather_blur_ksize"],
            blend_strength=CFG["local_blend_strength"]
        )

        residual_count, residual_max_conf = detect_residual_persons(fused_result)
        local_method = local_method + " + second_pass"

    total_time = t_global + t_local

    metrics = compute_region_metrics(
        reference_rgb=image,
        predicted_rgb=fused_result,
        region_mask=final_mask
    )

    return {
        "id": row["id"],
        "final_mask_area_ratio": float((final_mask > 0).sum()) / float(final_mask.shape[0] * final_mask.shape[1]),
        "local_method": local_method,
        "residual_count": residual_count,
        "residual_max_conf": residual_max_conf,
        "global_time_sec": t_global,
        "local_time_sec": t_local,
        "total_time_sec": total_time,
        "psnr_region": metrics["psnr_region"],
        "ssim_region": metrics["ssim_region"],
        "lpips_full": metrics["lpips_full"],
        "out_ssim": metrics["out_ssim"],
        "image": image,
        "mask": mask,
        "ring": ring,
        "final_mask": final_mask,
        "global_result": global_result,
        "fused_result": fused_result,
        "local_bbox": local_bbox,
        "second_pass_used": second_pass_used
    }


In [ ]:
# ============================================================
# CELL 10: Run improved pipeline on target split
# ============================================================

all_outputs = []
result_rows = []

for _, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Stage 2 improved inference"):
    try:
        out = process_one_sample(row)
        if out is None:
            continue

        sample_id = out["id"]

        Image.fromarray(out["image"]).save(STAGE2_IMG_ROOT / f"{sample_id}_input.png")
        Image.fromarray(out["mask"]).save(STAGE2_IMG_ROOT / f"{sample_id}_mask.png")
        Image.fromarray(out["ring"]).save(STAGE2_IMG_ROOT / f"{sample_id}_ring.png")
        Image.fromarray(out["final_mask"]).save(STAGE2_IMG_ROOT / f"{sample_id}_final_mask.png")
        Image.fromarray(out["global_result"]).save(STAGE2_IMG_ROOT / f"{sample_id}_global_lama.png")
        Image.fromarray(out["fused_result"]).save(STAGE2_IMG_ROOT / f"{sample_id}_final_result.png")

        result_rows.append({
            "id": sample_id,
            "final_mask_area_ratio": out["final_mask_area_ratio"],
            "local_method": out["local_method"],
            "residual_count": out["residual_count"],
            "residual_max_conf": out["residual_max_conf"],
            "global_time_sec": out["global_time_sec"],
            "local_time_sec": out["local_time_sec"],
            "total_time_sec": out["total_time_sec"],
            "psnr_region": out["psnr_region"],
            "ssim_region": out["ssim_region"],
            "lpips_full": out["lpips_full"],
            "out_ssim": out["out_ssim"],
            "second_pass_used": out["second_pass_used"],
            "feather_blur_ksize": CFG["feather_blur_ksize"],
            "mask_smooth_kernel": CFG["mask_smooth_kernel"]
        })

        all_outputs.append(out)

    except Exception as e:
        print(f"Error in sample {row['id']}: {e}")

results_df = pd.DataFrame(result_rows)
results_csv = STAGE2_META_ROOT / f"{EXPERIMENT_NAME}_{CFG['target_split']}.csv"
results_df.to_csv(results_csv, index=False)

print("Stage 2 improved inference complete.")
print("Saved results CSV:", results_csv)
display(results_df.head())


In [ ]:
# ============================================================
# CELL 11: Summary table
# ============================================================

if len(results_df) == 0:
    raise ValueError("No Stage 2 results found.")

summary = results_df[[
    "final_mask_area_ratio",
    "residual_count",
    "residual_max_conf",
    "global_time_sec",
    "local_time_sec",
    "total_time_sec",
    "psnr_region",
    "ssim_region",
    "lpips_full",
    "out_ssim"
]].describe()

print("Improved pipeline summary:")
display(summary)

print("Mean values:")
display(results_df.mean(numeric_only=True).to_frame("mean").T)


In [ ]:
# ============================================================
# CELL 12: Visual preview of outputs
# ============================================================

num_show = min(CFG["num_preview_samples"], len(all_outputs))

for i in range(num_show):
    out = all_outputs[i]

    plt.figure(figsize=(22, 10))

    plt.subplot(2, 3, 1)
    plt.imshow(out["image"])
    plt.title(f"Input\n{out['id']}")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(out["final_mask"], cmap="gray")
    plt.title("Final Removal Mask")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(out["global_result"])
    plt.title("Global LaMa Result")
    plt.axis("off")

    plt.subplot(2, 3, 4)
    plt.imshow(out["fused_result"])
    plt.title("Final Fused Result")
    plt.axis("off")

    overlay = out["image"].copy()
    overlay[out["final_mask"] > 0] = [255, 0, 0]
    plt.subplot(2, 3, 5)
    plt.imshow(overlay)
    plt.title("Input + Final Mask Overlay")
    plt.axis("off")

    x1, y1, x2, y2 = out["local_bbox"]
    local_vis = out["fused_result"].copy()
    cv2.rectangle(local_vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
    plt.subplot(2, 3, 6)
    plt.imshow(local_vis)
    plt.title(f"Local Box\nmethod={out['local_method']} | residual={out['residual_count']} | conf={out['residual_max_conf']:.3f}")
    plt.axis("off")

    plt.show()


# Important Interpretation Note

The current evaluation is conducted on real images in which the target person originally exists in the scene. Therefore, a true clean-background ground truth image is not available for direct reconstruction comparison.

For this reason:
- `residual_count` and `residual_max_conf` are treated as primary indicators of removal success
- `out_ssim` is used as a proxy for background preservation
- `psnr_region`, `ssim_region`, and `lpips_full` should be interpreted as proxy consistency metrics rather than absolute reconstruction metrics

For stronger paper-level evaluation in future experiments, the following should be added:
- a synthetic paired dataset
- true clean-background target images
- masked-region quantitative evaluation against ground-truth targets
